# Import necessary libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_curve, roc_curve, auc
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Step 1: Load the dataset
# Note: Replace 'transactions.csv' with your actual file path

In [2]:
df=pd.read_csv('/content/PS_20174392719_1491204439457_log.csv')

# Display basic info about the dataset

In [3]:
print("Dataset Info:")
print(df.info())
print("\nFirst 5 rows:")
print(df.head())
print("\nLast 5 rows:")
print(df.tail())

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 409488 entries, 0 to 409487
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   step            409488 non-null  int64  
 1   type            409488 non-null  object 
 2   amount          409487 non-null  float64
 3   nameOrig        409487 non-null  object 
 4   oldbalanceOrg   409487 non-null  float64
 5   newbalanceOrig  409487 non-null  float64
 6   nameDest        409487 non-null  object 
 7   oldbalanceDest  409487 non-null  float64
 8   newbalanceDest  409487 non-null  float64
 9   isFraud         409487 non-null  float64
 10  isFlaggedFraud  409487 non-null  float64
dtypes: float64(7), int64(1), object(3)
memory usage: 34.4+ MB
None

First 5 rows:
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295    

## Step 2: Data Cleaning and Preprocessing
# Drop unnecessary columns

In [4]:
df_clean = df.drop(['nameOrig', 'nameDest'], axis=1)

# One-hot encode the 'type' column

In [5]:
df_clean = pd.get_dummies(df_clean, columns=['type'], drop_first=True)

# Check for missing values

In [6]:
print("\nMissing values in each column:")
print(df_clean.isnull().sum())


Missing values in each column:
step              0
amount            1
oldbalanceOrg     1
newbalanceOrig    1
oldbalanceDest    1
newbalanceDest    1
isFraud           1
isFlaggedFraud    1
type_CASH_IN      0
type_CASH_OUT     0
type_DEBIT        0
type_PAYMENT      0
type_TRANSFER     0
dtype: int64


# Drop rows with missing values

In [7]:
df_clean = df_clean.dropna()

# Separate features and target

In [8]:
X = df_clean.drop('isFraud', axis=1)
y = df_clean['isFraud']

# Scale numerical features

In [9]:
scaler = StandardScaler()
num_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'step']
X[num_cols] = scaler.fit_transform(X[num_cols])

# Display cleaned data

In [10]:
print("\nCleaned dataset:")
print(X.head())


Cleaned dataset:
       step    amount  oldbalanceOrg  newbalanceOrig  oldbalanceDest  \
0 -3.630143 -0.570159      -0.247415       -0.254157       -0.418346   
1 -3.630143 -0.598153      -0.297653       -0.301115       -0.418346   
2 -3.630143 -0.604061      -0.304762       -0.307575       -0.418346   
3 -3.630143 -0.604061      -0.304762       -0.307575       -0.409342   
4 -3.630143 -0.563741      -0.290801       -0.297615       -0.418346   

   newbalanceDest  isFlaggedFraud  type_CASH_IN  type_CASH_OUT  type_DEBIT  \
0       -0.463398             0.0         False          False       False   
1       -0.463398             0.0         False          False       False   
2       -0.463398             0.0         False          False       False   
3       -0.463398             0.0         False           True       False   
4       -0.463398             0.0         False          False       False   

   type_PAYMENT  type_TRANSFER  
0          True          False  
1          Tru

## Step 3: Train-Test Split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Check class distribution

In [12]:
print("\nClass distribution in training set:")
print(y_train.value_counts(normalize=True))
print("\nClass distribution in test set:")
print(y_test.value_counts(normalize=True))


Class distribution in training set:
isFraud
0.0    0.99949
1.0    0.00051
Name: proportion, dtype: float64

Class distribution in test set:
isFraud
0.0    0.999487
1.0    0.000513
Name: proportion, dtype: float64


## Step 4: Handle Class Imbalance
# Calculate scale_pos_weight

In [13]:
fraud_count = y_train.sum()
non_fraud_count = len(y_train) - fraud_count
scale_pos_weight = non_fraud_count / fraud_count
print(f"\nScale_pos_weight: {scale_pos_weight:.2f}")


Scale_pos_weight: 1960.61


## Step 5: Initial XGBoost Model
# Initialize XGBoost classifier

In [14]:
xgb_model = XGBClassifier(objective='binary:logistic',
                         scale_pos_weight=scale_pos_weight,
                         random_state=42,
                         eval_metric='logloss')

# Train the model

In [15]:
xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

# Make predictions

In [16]:
y_pred_xgb = xgb_model.predict(X_test)

# Evaluate the model

In [17]:
print("\nInitial Model Evaluation:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))


Initial Model Evaluation:
Accuracy: 0.9997

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     81856
         1.0       0.65      0.79      0.71        42

    accuracy                           1.00     81898
   macro avg       0.82      0.89      0.85     81898
weighted avg       1.00      1.00      1.00     81898



## Step 6: Hyperparameter Tuning with RandomizedSearchCV
# Define parameter grid

In [18]:
param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [50, 100, 200],
    'scale_pos_weight': [scale_pos_weight]
}

# Initialize RandomizedSearchCV

In [19]:
random_search = RandomizedSearchCV(estimator=xgb_model,
                                 param_distributions=param_grid,
                                 n_iter=10,
                                 scoring='precision',
                                 cv=3,
                                 verbose=1,
                                 n_jobs=-1,
                                 random_state=42)

# Perform random search
random_search.fit(X_train, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


RandomizedSearchCV(cv=3,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=False,
                                           eval_metric='logloss',
                                           feature_types=None, gamma=None,
                                           grow_policy=None,
                                           importance_type=None,
                                           interaction_constraints=None,
                                           learning...
                                           min_child_weight=None, missing=nan,
                                           monotone_constraints=None,
                                           multi_strategy=None,
                                           n_estimators=None, n_jobs=None,
                                           num_parallel_tree=None,
                                           random_state=42, ...),
                   n_jobs=-1,
                   param_distributions={'learning_rate': [0.01, 0.1, 0.2],
                                        'max_depth': [4, 6, 8],
                                        'n_estimators': [50, 100, 200],
                                        'scale_pos_weight': [np.float64(1960.6107784431138)]},
                   random_state=42, scoring='precision', verbose=1)

# Get best parameters

In [20]:
best_params = random_search.best_params_
print("\nBest hyperparameters:")
print(best_params)


Best hyperparameters:
{'scale_pos_weight': np.float64(1960.6107784431138), 'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.1}


## Step 7: Train Model with Best Hyperparameters
# Initialize model with best parameters

In [21]:
xgb_best_model = XGBClassifier(objective='binary:logistic',
                              scale_pos_weight=best_params['scale_pos_weight'],
                              n_estimators=best_params['n_estimators'],
                              max_depth=best_params['max_depth'],
                              learning_rate=best_params['learning_rate'],
                              random_state=42,
                              eval_metric='logloss')

# Train the model

In [22]:
xgb_best_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

# Make predictions

In [23]:
y_pred_xgb_best = xgb_best_model.predict(X_test)

# Evaluate the model

In [24]:
print("\nOptimized Model Evaluation:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb_best):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb_best))


Optimized Model Evaluation:
Accuracy: 0.9997

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     81856
         1.0       0.71      0.76      0.74        42

    accuracy                           1.00     81898
   macro avg       0.86      0.88      0.87     81898
weighted avg       1.00      1.00      1.00     81898



## Step 8: Feature Importance Analysis
# Get feature importances

In [25]:
importances = xgb_best_model.feature_importances_
feature_names = X.columns

# Create importance DataFrame

In [26]:
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values('Importance', ascending=True)

# Plot feature importance

In [27]:
fig = px.bar(importance_df,
             x='Importance',
             y='Feature',
             title='Feature Importance',
             labels={'Importance': 'Importance Score', 'Feature': 'Feature'},
             height=600)

fig.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig.show()
fig.write_html("feature_importance_plot.html")

## Step 9: Confusion Matrix Visualization
# Compute confusion matrix

In [28]:
cm = confusion_matrix(y_test, y_pred_xgb_best)

In [29]:
print(cm)

[[81843    13]
 [   10    32]]


# Create heatmap

In [30]:
fig = go.Figure(data=go.Heatmap(
                   z=cm,
                   x=['Not Fraud', 'Fraud'],
                   y=['Not Fraud', 'Fraud'],
                   colorscale='Blues',
                   text=cm,
                   texttemplate="%{text}",
                   hoverinfo="text"))

# Add annotations

In [31]:
annotations = []
for i, row in enumerate(cm):
    for j, value in enumerate(row):
        annotations.append(
            go.layout.Annotation(
                text=str(value),
                x=['Not Fraud', 'Fraud'][j],
                y=['Not Fraud', 'Fraud'][i],
                showarrow=False,
                font=dict(color='black' if value < cm.max()/2 else 'white')
            )
        )


In [34]:
fig.update_layout(
    title='Confusion Matrix',
    xaxis_title='Predicted',
    yaxis_title='Actual',
    annotations=annotations,
    width=600,
    height=600
)
fig.show()
fig.write_html("confusion_matrix.html")

## Step 10: Threshold Optimization
# Get predicted probabilities

In [35]:
y_pred_prob_xgb_best = xgb_best_model.predict_proba(X_test)[:, 1]

# Test different thresholds

In [36]:
thresholds = np.arange(0.5, 0.96, 0.05)

In [37]:
print(thresholds)

[0.5  0.55 0.6  0.65 0.7  0.75 0.8  0.85 0.9  0.95]


In [39]:
results = []

for threshold in thresholds:
    y_pred_thresh = (y_pred_prob_xgb_best >= threshold).astype(int)
    report = classification_report(y_test, y_pred_thresh, output_dict=True)
    precision = report['1.0']['precision']
    recall = report['1.0']['recall']
    f1 = report['1.0']['f1-score']
    results.append({'Threshold': threshold, 'Precision': precision, 'Recall': recall, 'F1': f1})

results_df = pd.DataFrame(results)

# Plot threshold analysis

In [41]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=results_df['Threshold'], y=results_df['Precision'],
                    mode='lines+markers',
                    name='Precision',
                    line=dict(color='blue')))

fig.add_trace(go.Scatter(x=results_df['Threshold'], y=results_df['Recall'],
                    mode='lines+markers',
                    name='Recall',
                    line=dict(color='green')))

fig.add_trace(go.Scatter(x=results_df['Threshold'], y=results_df['F1'],
                    mode='lines+markers',
                    name='F1-score',
                    line=dict(color='orange')))


Highlight Choosen Threshold 0.675

In [43]:
# Assuming results_df is already calculated
import numpy as np
chosen_threshold = 0.675  # Or any other desired value

# Find the closest threshold in results_df
closest_threshold = results_df.loc[(np.abs(results_df['Threshold'] - chosen_threshold)).idxmin(), 'Threshold']

# Get the index of the closest threshold
chosen_idx = results_df[results_df['Threshold'] == closest_threshold].index[0]

# Now you can use chosen_idx safely
fig.add_annotation(x=closest_threshold, y=results_df.loc[chosen_idx, 'Precision'],
                  text=f"Threshold: {closest_threshold}<br>Precision: {results_df.loc[chosen_idx, 'Precision']:.2f}<br>Recall: {results_df.loc[chosen_idx, 'Recall']:.2f}",
                  showarrow=True,
                  arrowhead=1)

fig.update_layout(title='Precision, Recall, and F1-score at Different Thresholds',
                 xaxis_title='Threshold',
                 yaxis_title='Score',
                 plot_bgcolor='white',
                 paper_bgcolor='white',
                 width=900,
                 height=600)

fig.show()
fig.write_html("thresholds_precision_recall_f1.html")

In [56]:
# Add a vertical line at the chosen threshold
fig.add_vline(x=closest_threshold, line_dash="dash", line_color="red")

# Reposition annotation to avoid overlap
fig.update_annotations(dict(
    xref="x", yref="y",
    yanchor="bottom", xanchor="left",
    font_size=12
))

In [57]:
import plotly.graph_objects as go

# Create the base plot
fig = go.Figure()

# Add traces for Precision, Recall, F1
fig.add_trace(go.Scatter(
    x=results_df['Threshold'],
    y=results_df['Precision'],
    mode='lines+markers',
    name='Precision',
    line=dict(color='blue', width=2)
))

fig.add_trace(go.Scatter(
    x=results_df['Threshold'],
    y=results_df['Recall'],
    mode='lines+markers',
    name='Recall',
    line=dict(color='green', width=2)
))

fig.add_trace(go.Scatter(
    x=results_df['Threshold'],
    y=results_df['F1'],
    mode='lines+markers',
    name='F1-score',
    line=dict(color='orange', width=2)
))

# Add vertical line at chosen threshold
fig.add_vline(
    x=chosen_threshold,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Optimal Threshold: {chosen_threshold}",
    annotation_position="top right"
)

# Add annotation with metrics
fig.add_annotation(
    x=chosen_threshold,
    y=results_df.loc[chosen_idx, 'Precision'],
    text=f"<b>Threshold {chosen_threshold}</b><br>Precision: {results_df.loc[chosen_idx, 'Precision']:.2f}<br>Recall: {results_df.loc[chosen_idx, 'Recall']:.2f}",
    showarrow=True,
    arrowhead=1,
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
    font=dict(size=12)
)

# Update layout for clarity
fig.update_layout(
    title='<b>Fraud Detection: Precision-Recall Trade-off</b><br><sub>Higher thresholds reduce false positives but miss more fraud</sub>',
    xaxis_title='Decision Threshold',
    yaxis_title='Score',
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=900,
    height=600,
    legend=dict(orientation="h", yanchor="bottom", y=1.02)
)

fig.show()
fig.write_html("thresholds_precision_recall_f1_enhanced.html")

## Step 11: Precision-Recall Curve

In [44]:
precision, recall, thresholds_pr = precision_recall_curve(y_test, y_pred_prob_xgb_best)

# Find threshold closest to 0.675

In [47]:
optimal_threshold_idx = np.argmin(np.abs(thresholds_pr - chosen_threshold))

In [48]:
fig = go.Figure()

In [49]:
fig.add_trace(go.Scatter(x=recall, y=precision,
                        mode='lines',
                        name='Precision-Recall Curve',
                        line=dict(color='blue')))


# Add optimal threshold point

In [50]:
fig.add_trace(go.Scatter(x=[recall[optimal_threshold_idx]],
                        y=[precision[optimal_threshold_idx]],
                        mode='markers',
                        name=f'Threshold = {chosen_threshold}',
                        marker=dict(color='red', size=10)))

fig.update_layout(title='Precision-Recall Curve',
                 xaxis_title='Recall',
                 yaxis_title='Precision',
                 plot_bgcolor='white',
                 paper_bgcolor='white',
                 width=800,
                 height=600)

fig.show()
fig.write_html("precision_recall_curve.html")

## Step 12: ROC Curve

In [51]:
fpr, tpr, thresholds_roc = roc_curve(y_test, y_pred_prob_xgb_best)
roc_auc = auc(fpr, tpr)

fig = go.Figure()

In [52]:
fig.add_trace(go.Scatter(x=fpr, y=tpr,
                        mode='lines',
                        name=f'ROC curve (AUC = {roc_auc:.2f})',
                        line=dict(color='orange')))

fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1],
                        mode='lines',
                        name='Random Classifier',
                        line=dict(color='navy', dash='dash')))

fig.update_layout(title='Receiver Operating Characteristic (ROC) Curve',
                 xaxis_title='False Positive Rate',
                 yaxis_title='True Positive Rate',
                 plot_bgcolor='white',
                 paper_bgcolor='white',
                 width=800,
                 height=600)

fig.show()
fig.write_html("roc_curve.html")


## Step 13: Final Model with Chosen Threshold
# Apply chosen threshold

In [53]:
y_pred_adjusted = (y_pred_prob_xgb_best >= chosen_threshold).astype(int)

# Evaluate final model

In [55]:
print("\nFinal Model Evaluation (Threshold = 0.675):")
print(f"Accuracy: {accuracy_score(y_test, y_pred_adjusted):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_adjusted))


Final Model Evaluation (Threshold = 0.675):
Accuracy: 0.9998

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     81856
         1.0       0.76      0.76      0.76        42

    accuracy                           1.00     81898
   macro avg       0.88      0.88      0.88     81898
weighted avg       1.00      1.00      1.00     81898

